# PET sensitivity and the temperature effect on SPEI

Answers Reviewer 1 (comment 9), Reviewer 2 (comment 7) and Reviewer 3 (comment 2).

**Needs in the same folder:** `reshaped_climate_data.xlsx` and `extra_variables.xlsx` (Tmax, Tmin, RH, Wind, Srad, Elevation sheets).

**PET formulations**
| Formulation | Inputs | Window |
|---|---|---|
| Thornthwaite (with d/30) | mean temperature | A and B |
| Hamon | mean temperature, day length | A and B |
| Oudin | mean temperature, extraterrestrial radiation | A and B |
| Hargreaves-Samani | Tmax, Tmin, extraterrestrial radiation | A and B |
| FAO-56 Penman-Monteith | Tmax, Tmin, RH, wind, solar radiation, elevation | **B only** |

**Two analysis windows.** CERES solar radiation starts in 1984, so Penman-Monteith cannot be computed for 1981-1983.
- **Window A:** 1982-2001 vs 2002-2021 (the main analysis window) for the four formulations that do not need radiation.
- **Window B:** 1985-2002 vs 2003-2020 (SPEI fitted on 1984 onward) for **all five** formulations, so Penman-Monteith is compared like-for-like.

**Three index variants per formulation** (all with the same rank-based standardisation, so only the PET differs):
1. `P only` - precipitation alone (an SPI-like index)
2. `climatological PET` - PET replaced by its calendar-month mean (removes PET variability and trend)
3. `actual PET` - the SPEI

Differences between 2 and 3 isolate the effect of the PET (temperature) trend on the SH-ITA results.

In [1]:
import os, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display
from math import sin, radians, acos, tan, pi
from scipy import stats

# ---- EDIT THESE TWO LINES IF NEEDED -------------------------------------------
DATA = 'reshaped_climate_data.xlsx'   # the workbook your first notebook created (same folder as this notebook)
OUT  = 'sh_ita_outputs'               # folder where tables and figures will be saved
# ------------------------------------------------------------------------------
os.makedirs(OUT, exist_ok=True)
%matplotlib inline

## Settings

In [2]:
LAT = {'YENAGOA': 4.95, 'IKOT AKPADEN': 4.62, 'IBADAN': 7.38, 'ABUJA': 9.07,
       'JALINGO': 8.89, 'SOKOTO': 13.01, 'GOMBE': 10.28, 'KUKAWA': 12.92}
X_START, X_END = '1982-01-01', '2001-12-31'      # first half  (240 months)
Y_START, Y_END = '2002-01-01', '2021-12-31'      # second half (240 months)
B = 2000                                          # bootstrap replicates
BLOCK = {3: 12, 12: 24}                           # block length (months) by timescale
MIN_N = 10                                        # min class size for a trend label
SEED = 2026
CLIP = 1e-6
COL = {'Low': 'red', 'Medium': 'orange', 'High': 'blue'}   # ONE colour convention for every figure

## Load data

In [3]:
P = pd.read_excel(DATA, 'Precip_Wide', index_col='Date').drop(columns=['Year', 'Month'], errors='ignore')
T = pd.read_excel(DATA, 'Temp_Wide', index_col='Date').drop(columns=['Year', 'Month'], errors='ignore')
P.index = pd.to_datetime(P.index); T.index = pd.to_datetime(T.index)

## PET (Thornthwaite)

In [4]:
MID = [15, 46, 75, 105, 135, 166, 196, 227, 258, 288, 319, 349]
def daylength(lat, month):
    decl = radians(23.45 * sin(radians(360 * (284 + MID[month - 1]) / 365.0)))
    x = max(min(-tan(radians(lat)) * tan(decl), 0.999999), -0.999999)
    return 24.0 / pi * acos(x)
def thornthwaite(t, lat):
    clim = t.groupby(t.index.month).mean()
    I = ((clim.clip(lower=0) / 5.0) ** 1.514).sum()
    a = 6.75e-7 * I ** 3 - 7.71e-5 * I ** 2 + 1.792e-2 * I + 0.49239
    out = [16.0 * (10.0 * v / I) ** a * (daylength(lat, d.month) / 12.0) * (d.days_in_month / 30.0)
           if v > 0 else 0.0 for d, v in t.items()]
    return pd.Series(out, index=t.index)
PET = pd.DataFrame({l: thornthwaite(T[l], LAT[l]) for l in P.columns})

## SPI and SPEI per calendar month

In [5]:
def roll(s, sc): return s.rolling(sc, min_periods=sc).sum()
def spi_cm(p, sc):
    a = roll(p, sc); out = pd.Series(np.nan, index=a.index)
    for m in range(1, 13):
        v = a[a.index.month == m].dropna(); nz = v[v > 0]
        if len(nz) < 10: continue
        q = (v == 0).mean()
        g, _, sc_ = stats.gamma.fit(nz.values, floc=0)
        cdf = np.where(v.values <= 0, q, q + (1 - q) * stats.gamma.cdf(v.values, g, loc=0, scale=sc_))
        out[v.index] = stats.norm.ppf(np.clip(cdf, CLIP, 1 - CLIP))
    return out
def spei_cm(p, pet, sc):
    a = roll(p - pet, sc); out = pd.Series(np.nan, index=a.index)
    for m in range(1, 13):
        v = a[a.index.month == m].dropna(); N = len(v)
        if N < 20: continue
        out[v.index] = stats.norm.ppf((stats.rankdata(v.values) - 0.44) / (N + 0.12))
    return out

## SH-ITA statistics and bootstrap

In [6]:
def slope(x, y):
    return np.nan if len(x) < 2 or np.ptp(x) == 0 else np.polyfit(x, y, 1)[0]
def ita_stats(x, y):
    xs, ys = np.sort(x), np.sort(y)
    cls = np.where(xs <= -1, 0, np.where(xs >= 1, 2, 1))          # class from FIRST-half value
    r = {}
    for k, nm in enumerate(['low', 'med', 'high']):
        m = cls == k
        r['n_' + nm] = int(m.sum()); r['slope_' + nm] = slope(xs[m], ys[m])
        r['D_' + nm] = (ys[m] - xs[m]).mean() if m.any() else np.nan
    r['IISC'] = (ys.mean() - xs.mean()) / np.abs(xs).mean()
    sl, sh = r['slope_low'], r['slope_high']
    r['ICSC'] = (sh - sl) / abs(sh + sl) if np.isfinite(sl) and np.isfinite(sh) and abs(sh + sl) > 0 else np.nan
    r['A'] = r['D_high'] - r['D_low']
    return r, xs, ys, cls
def cbb(a, L, rng):
    n = len(a); nb = int(np.ceil(n / L))
    return a[(rng.integers(0, n, nb)[:, None] + np.arange(L)).ravel()[:n] % n]
KEYS = ['D_low', 'D_med', 'D_high', 'IISC', 'ICSC', 'A']
def analyse(s, L, seed):
    x = s.loc[X_START:X_END].dropna().values; y = s.loc[Y_START:Y_END].dropna().values
    assert len(x) == len(y), 'halves differ in length: %d vs %d' % (len(x), len(y))
    base, xs, ys, cls = ita_stats(x, y); rng = np.random.default_rng(seed)
    bt = {k: [] for k in KEYS}
    for _ in range(B):
        r, *_ = ita_stats(cbb(x, L, rng), cbb(y, L, rng))
        for k in KEYS: bt[k].append(r[k])
    res = dict(base); res['n'] = len(x)
    for k in KEYS:
        v = np.array(bt[k], float); v = v[np.isfinite(v)]
        res[k + '_lo'], res[k + '_hi'] = (np.percentile(v, 2.5), np.percentile(v, 97.5)) if len(v) > 100 else (np.nan, np.nan)
    for nm in ['low', 'med', 'high']:
        lo, hi = res['D_%s_lo' % nm], res['D_%s_hi' % nm]
        res['label_' + nm] = '-' if (res['n_' + nm] < MIN_N or not np.isfinite(lo)) else ('Yes(+)' if lo > 0 else 'Yes(-)' if hi < 0 else 'No')
    return res, xs, ys, cls

## Comparators: Mann-Kendall (Hamed-Rao), Sen (2017), Alashan (2020)

In [7]:
def sen_slope(x):
    x = np.asarray(x, float); n = len(x)
    return np.median(np.concatenate([(x[k:] - x[:-k]) / k for k in range(1, n)]))

def mk_test(x, hr=True):
    """Mann-Kendall S, Z, p-value (two-sided) and Sen slope per time step. hr=True applies Hamed & Rao (1998)."""
    x = np.asarray(x, float); n = len(x)
    S = sum(np.sign(x[i + 1:] - x[i]).sum() for i in range(n - 1))
    _, cnt = np.unique(x, return_counts=True)
    var = (n * (n - 1) * (2 * n + 5) - np.sum(cnt * (cnt - 1) * (2 * cnt + 5))) / 18.0
    beta = sen_slope(x); f = 1.0
    if hr:
        r = stats.rankdata(x - beta * np.arange(n)); r = r - r.mean(); den = (r * r).sum()
        lim = stats.norm.ppf(0.975) / np.sqrt(n); sni = 0.0
        for k in range(1, n - 1):
            a = (r[:-k] * r[k:]).sum() / den
            if abs(a) > lim: sni += (n - k) * (n - k - 1) * (n - k - 2) * a
        f = 1 + 2.0 / (n * (n - 1) * (n - 2)) * sni
        f = f if f > 0 else 1.0
    z = 0.0 if S == 0 else (S - np.sign(S)) / np.sqrt(var * f)
    return dict(S=S, Z=z, p=2 * (1 - stats.norm.cdf(abs(z))), sen=beta, vcf=f)

def direction(p, est, alpha=0.05):
    return '0' if not (p < alpha) else ('+' if est > 0 else '-')

## Final analysis function (bootstrap p-values + Holm correction)

In [8]:
BL = {3: (6, 12, 24), 12: (12, 24, 48)}       # block lengths tried; the middle one is primary
ALPHA = 0.05

def boot_p(v):
    """Two-sided percentile-bootstrap p-value for H0: statistic = 0."""
    v = v[np.isfinite(v)]
    return np.nan if len(v) < 100 else max(2 * min((v <= 0).mean(), (v >= 0).mean()), 1.0 / (len(v) + 1))

def holm(pvals):
    """Holm step-down adjustment; NaN entries (untestable classes) are ignored."""
    p = np.array(pvals, float); out = np.full_like(p, np.nan); ok = np.where(np.isfinite(p))[0]
    order = ok[np.argsort(p[ok])]; m = len(order); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * p[i])); out[i] = run
    return out

def lab(D, p, n):
    return '-' if (n < MIN_N or not np.isfinite(p)) else ('No' if p >= ALPHA else ('Yes(+)' if D > 0 else 'Yes(-)'))

def analyse_final(s, L, seed, B=B):
    x = s.loc[X_START:X_END].dropna().values; y = s.loc[Y_START:Y_END].dropna().values
    assert len(x) == len(y)
    base, xs, ys, cls = ita_stats(x, y); rng = np.random.default_rng(seed)
    bt = {k: [] for k in KEYS}
    for _ in range(B):
        r, *_ = ita_stats(cbb(x, L, rng), cbb(y, L, rng))
        for k in KEYS: bt[k].append(r[k])
    bt = {k: np.array(v, float) for k, v in bt.items()}
    res = dict(base, n=len(x), L=L)
    for k in KEYS:
        v = bt[k][np.isfinite(bt[k])]
        res[k + '_lo'], res[k + '_hi'] = (np.percentile(v, 2.5), np.percentile(v, 97.5)) if len(v) > 100 else (np.nan, np.nan)
    names = ['low', 'med', 'high']
    p_raw = [boot_p(bt['D_' + nm]) if base['n_' + nm] >= MIN_N else np.nan for nm in names]
    p_adj = holm(p_raw)
    for i, nm in enumerate(names):
        res['p_raw_' + nm] = p_raw[i]; res['p_holm_' + nm] = p_adj[i]
        res['label_raw_' + nm] = lab(base['D_' + nm], p_raw[i], base['n_' + nm])
        res['label_holm_' + nm] = lab(base['D_' + nm], p_adj[i], base['n_' + nm])
    for k in ('IISC', 'ICSC', 'A'):
        res[k + '_clear'] = bool(np.isfinite(res[k + '_lo']) and (res[k + '_lo'] > 0 or res[k + '_hi'] < 0))
    return res

## PET formulations

In [9]:
def Ra_daily(lat_deg, doy):
    """Extraterrestrial radiation, MJ m-2 day-1 (FAO-56)."""
    phi = np.radians(lat_deg); dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365.0)
    dec = 0.409 * np.sin(2 * np.pi * doy / 365.0 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(dec), -1, 1))
    return (24 * 60 / np.pi) * 0.0820 * dr * (ws * np.sin(phi) * np.sin(dec) + np.cos(phi) * np.cos(dec) * np.sin(ws))
def es_kpa(t): return 0.6108 * np.exp(17.27 * t / (t + 237.3))
def monthly_from_daily(daily, idx): return pd.Series(np.asarray(daily) * idx.days_in_month.values, index=idx)
def doy_mid(idx): return np.array([pd.Timestamp(d.year, d.month, 15).dayofyear for d in idx])

def pet_hamon(t, lat, kpec=1.0):
    """Hamon (1961): PET[mm/day] = 0.1651 * (N/12) * 216.7 * esat[mb]/(T+273.3) * kpec."""
    N = np.array([daylength(lat, d.month) for d in t.index])
    rhosat = 216.7 * 6.108 * np.exp(17.27 * t.values / (t.values + 237.3)) / (t.values + 273.3)
    return monthly_from_daily(np.where(t.values > 0, 0.1651 * (N / 12.0) * rhosat * kpec, 0.0), t.index)
def pet_oudin(t, lat):
    """Oudin et al. (2005): PET[mm/day] = 0.408 * Ra * (T + 5) / 100 if T + 5 > 0."""
    ra = Ra_daily(lat, doy_mid(t.index))
    return monthly_from_daily(np.where(t.values + 5 > 0, 0.408 * ra * (t.values + 5) / 100.0, 0.0), t.index)
def pet_hargreaves(tmax, tmin, lat):
    """Hargreaves-Samani: PET[mm/day] = 0.0023 * 0.408*Ra * (Tmean + 17.8) * sqrt(Tmax - Tmin), Tmean = (Tmax+Tmin)/2."""
    tm = (tmax + tmin) / 2; ra = Ra_daily(lat, doy_mid(tm.index))
    return monthly_from_daily(0.0023 * 0.408 * ra * (tm.values + 17.8) * np.sqrt(np.clip(tmax.values - tmin.values, 0, None)), tm.index)
def pet_pm_fao56(tmax, tmin, rh, u2, rs, lat, elev):
    """FAO-56 Penman-Monteith on monthly mean data (G = 0). rs in MJ m-2 day-1, u2 m/s at 2 m, rh %."""
    Tx, Tn = tmax.values, tmin.values; T = (Tx + Tn) / 2
    Pk = 101.3 * ((293 - 0.0065 * elev) / 293) ** 5.26; gamma = 0.000665 * Pk
    delta = 4098 * es_kpa(T) / (T + 237.3) ** 2
    es = (es_kpa(Tx) + es_kpa(Tn)) / 2; ea = np.clip(rh.values, 1, 100) / 100.0 * es
    ra = Ra_daily(lat, doy_mid(tmax.index)); rso = (0.75 + 2e-5 * elev) * ra
    rn = 0.77 * rs.values - 4.903e-9 * (((Tx + 273.16) ** 4 + (Tn + 273.16) ** 4) / 2) * (0.34 - 0.14 * np.sqrt(ea)) * (1.35 * np.clip(rs.values / rso, 0.3, 1.0) - 0.35)
    et0 = (0.408 * delta * rn + gamma * 900 / (T + 273) * u2.values * (es - ea)) / (delta + gamma * (1 + 0.34 * u2.values))
    return monthly_from_daily(np.clip(et0, 0, None), tmax.index)

PETS = {'Thornthwaite': PET.copy(),
        'Hamon': pd.DataFrame({l: pet_hamon(T[l], LAT[l]) for l in P.columns}),
        'Oudin': pd.DataFrame({l: pet_oudin(T[l], LAT[l]) for l in P.columns})}
XV = 'extra_variables.xlsx'
if os.path.exists(XV):
    ex = {s: pd.read_excel(XV, s, index_col='Date') for s in ('Tmax', 'Tmin', 'RH', 'Wind', 'Srad')}
    ELEV = pd.read_excel(XV, 'Elevation').set_index('Location')['Elevation_m'].to_dict()
    PETS['Hargreaves'] = pd.DataFrame({l: pet_hargreaves(ex['Tmax'][l], ex['Tmin'][l], LAT[l]) for l in P.columns})
    PETS['Penman-Monteith'] = pd.DataFrame({l: pet_pm_fao56(ex['Tmax'][l], ex['Tmin'][l], ex['RH'][l], ex['Wind'][l], ex['Srad'][l], LAT[l], ELEV[l]) for l in P.columns})
else:
    print('extra_variables.xlsx not found: Hargreaves and Penman-Monteith skipped.')
print('PET formulations:', list(PETS))

PET formulations: ['Thornthwaite', 'Hamon', 'Oudin', 'Hargreaves', 'Penman-Monteith']


## How different are the PET formulations? (1985-2020, all formulations available)

In [10]:
ref = 'Penman-Monteith' if 'Penman-Monteith' in PETS else 'Thornthwaite'
rows = []
for name, df in PETS.items():
    for loc in P.columns:
        d = df[loc].loc['1985':'2020']; ann = d.groupby(d.index.year).sum(); m = mk_test(ann.values, hr=True)
        rows.append(dict(PET=name, Location=loc.title(), mean_annual_mm=ann.mean(), r_monthly_with_reference=np.corrcoef(d, PETS[ref][loc].loc['1985':'2020'])[0, 1],
                         annual_trend_mm_per_decade=m['sen'] * 10, trend_p_MMK=m['p']))
PETCMP = pd.DataFrame(rows).round(3)
print('reference for correlations:', ref)
print(PETCMP.pivot(index='Location', columns='PET', values='mean_annual_mm').round(0).to_string())
print(PETCMP.pivot(index='Location', columns='PET', values='annual_trend_mm_per_decade').round(1).to_string())

reference for correlations: Penman-Monteith
PET            Hamon  Hargreaves   Oudin  Penman-Monteith  Thornthwaite
Location                                                               
Abuja         1424.0      2038.0  1609.0           1622.0        1462.0
Gombe         1593.0      2310.0  1705.0           2312.0        1942.0
Ibadan        1401.0      1784.0  1605.0           1275.0        1394.0
Ikot Akpaden  1569.0      1312.0  1721.0           1217.0        1785.0
Jalingo       1555.0      2187.0  1693.0           1916.0        1792.0
Kukawa        2007.0      2629.0  1908.0           3273.0        4428.0
Sokoto        1722.0      2448.0  1762.0           2482.0        2478.0
Yenagoa       1456.0      1655.0  1649.0           1170.0        1507.0
PET           Hamon  Hargreaves  Oudin  Penman-Monteith  Thornthwaite
Location                                                             
Abuja          52.7        56.5   34.9             38.2         124.5
Gombe          22.0       

## SH-ITA with each PET, and the temperature (PET) effect

In [11]:
NB = 1000        # bootstrap replicates for this sensitivity run
WINDOWS = {'A (1982-2021)': dict(start='1981-01-01', x=('1982-01-01', '2001-12-31'), y=('2002-01-01', '2021-12-31'),
                                 pets=['Thornthwaite', 'Hamon', 'Oudin', 'Hargreaves']),
           'B (1985-2020)': dict(start='1984-01-01', x=('1985-01-01', '2002-12-31'), y=('2003-01-01', '2020-12-31'),
                                 pets=['Thornthwaite', 'Hamon', 'Oudin', 'Hargreaves', 'Penman-Monteith'])}
def run_window(wname, cfg):
    global X_START, X_END, Y_START, Y_END
    X_START, X_END = cfg['x']; Y_START, Y_END = cfg['y']
    names = [n for n in cfg['pets'] if n in PETS]; out, comp = [], []
    for sc in (3, 12):
        L = BL[sc][1]
        for loc in P.columns:
            p = P[loc].loc[cfg['start']:]
            s0 = spei_cm(p, pd.Series(0.0, index=p.index), sc).loc[X_START:Y_END].dropna()
            r = analyse_final(s0, L, SEED, B=NB); r.update(window=wname, PET='-', variant='P only', location=loc.title(), scale=sc); out.append(r)
            for n in names:
                pet = PETS[n][loc].loc[cfg['start']:]; petc = pet.groupby(pet.index.month).transform('mean')
                sA, sC = spei_cm(p, pet, sc), spei_cm(p, petc, sc)
                for var, ser in (('actual PET', sA), ('climatological PET', sC)):
                    s = ser.loc[X_START:Y_END].dropna(); r = analyse_final(s, L, SEED, B=NB)
                    r.update(window=wname, PET=n, variant=var, location=loc.title(), scale=sc); out.append(r)
                dd = (sA - sC).loc[X_START:Y_END].dropna(); m = mk_test(dd.values, hr=True)
                comp.append(dict(window=wname, PET=n, location=loc.title(), scale=sc, PET_component_trend_per_decade=m['sen'] * 120, PET_component_sig=m['p'] < 0.05))
        print('done', wname, 'scale', sc, flush=True)
    return out, comp
res, comps = [], []
for w, cfg in WINDOWS.items():
    o, c = run_window(w, cfg); res += o; comps += c
SENS = pd.DataFrame(res); COMP = pd.DataFrame(comps)
SENS['IISC_dir'] = np.where(SENS.IISC_clear, np.where(SENS.IISC > 0, '+', '-'), '0')
print(len(SENS), 'analyses')

done A (1982-2021) scale 3
done A (1982-2021) scale 12
done B (1985-2020) scale 3
done B (1985-2020) scale 12
320 analyses


## Does the choice of PET change the SH-ITA conclusions?

In [12]:
act = SENS[SENS.variant == 'actual PET']
agree = []
for (w, loc, sc), g in act.groupby(['window', 'location', 'scale']):
    agree.append(dict(window=w, location=loc, scale=sc, IISC_dir_all_agree=g.IISC_dir.nunique() == 1,
                      IISC_dir=' / '.join(f'{p[:5]}:{d}' for p, d in zip(g.PET, g.IISC_dir)),
                      IISC_range=round(g.IISC.max() - g.IISC.min(), 2),
                      holm_labels_all_agree=all(g['label_holm_' + n].nunique() == 1 for n in ('low', 'med', 'high'))))
AGR = pd.DataFrame(agree)
for w, g in AGR.groupby('window'):
    print(f'Window {w}: IISC direction/significance identical across PET formulations in {int(g.IISC_dir_all_agree.sum())}/{len(g)}; Holm labels identical in {int(g.holm_labels_all_agree.sum())}/{len(g)}')
print('\nCases where formulations disagree on the overall shift:')
print(AGR[~AGR.IISC_dir_all_agree][['window', 'location', 'scale', 'IISC_dir']].to_string(index=False))

Window A (1982-2021): IISC direction/significance identical across PET formulations in 13/16; Holm labels identical in 11/16
Window B (1985-2020): IISC direction/significance identical across PET formulations in 10/16; Holm labels identical in 10/16

Cases where formulations disagree on the overall shift:
       window location  scale                                        IISC_dir
A (1982-2021)    Gombe     12           Thorn:- / Hamon:0 / Oudin:0 / Hargr:0
A (1982-2021)   Ibadan      3           Thorn:0 / Hamon:0 / Oudin:0 / Hargr:+
A (1982-2021)   Sokoto     12           Thorn:0 / Hamon:+ / Oudin:+ / Hargr:+
B (1985-2020)    Gombe      3 Thorn:- / Hamon:- / Oudin:- / Hargr:- / Penma:0
B (1985-2020)    Gombe     12 Thorn:- / Hamon:- / Oudin:0 / Hargr:0 / Penma:0
B (1985-2020)   Sokoto      3 Thorn:0 / Hamon:0 / Oudin:0 / Hargr:0 / Penma:+
B (1985-2020)   Sokoto     12 Thorn:0 / Hamon:+ / Oudin:+ / Hargr:+ / Penma:+
B (1985-2020)  Yenagoa      3 Thorn:- / Hamon:0 / Oudin:0 / Hargr:0 /

## How much of the SPEI shift comes from PET (temperature)?

In [13]:
piv = SENS.pivot_table(index=['window', 'location', 'scale', 'PET'], columns='variant', values='IISC').reset_index()
pOnly = SENS[SENS.variant == 'P only'].set_index(['window', 'location', 'scale']).IISC
piv['IISC_P_only'] = [pOnly[(w, l, s)] for w, l, s in zip(piv.window, piv.location, piv.scale)]
piv = piv[piv.PET != '-'].rename(columns={'actual PET': 'IISC_actual_PET', 'climatological PET': 'IISC_clim_PET'})
piv['PET_effect_on_IISC'] = piv.IISC_actual_PET - piv.IISC_clim_PET
piv = piv.merge(COMP, on=['window', 'PET', 'location', 'scale'])
print('Median |IISC change| due to PET (actual minus climatological PET):')
print(piv.groupby(['window', 'PET']).PET_effect_on_IISC.apply(lambda s: round(s.abs().median(), 2)).unstack().to_string())
print('\nShare of location x timescale cases where the PET component has a significant trend:')
print(piv.groupby(['window', 'PET']).PET_component_sig.mean().round(2).unstack().to_string())
piv.round(3)

Median |IISC change| due to PET (actual minus climatological PET):
PET            Hamon  Hargreaves  Oudin  Penman-Monteith  Thornthwaite
window                                                                
A (1982-2021)   0.21        0.23   0.14              NaN          0.29
B (1985-2020)   0.14        0.21   0.14              0.1          0.21

Share of location x timescale cases where the PET component has a significant trend:
PET            Hamon  Hargreaves  Oudin  Penman-Monteith  Thornthwaite
window                                                                
A (1982-2021)   0.88        0.56   0.81              NaN          0.94
B (1985-2020)   0.81        0.50   0.75             0.56          0.81


,window,location,scale,PET,P only,IISC_actual_PET,IISC_clim_PET,IISC_P_only,PET_effect_on_IISC,PET_component_trend_per_decade,PET_component_sig
0,A (1982-2021),Abuja,3,Hamon,NaN,-1.229,-0.886,-0.886,-0.342,-0.028,True
1,A (1982-2021),Abuja,3,Hargreaves,NaN,-1.225,-0.886,-0.886,-0.339,-0.040,True
2,A (1982-2021),Abuja,3,Oudin,NaN,-1.203,-0.889,-0.886,-0.314,0.000,True
3,A (1982-2021),Abuja,3,Thornthwaite,NaN,-1.299,-0.886,-0.886,-0.413,-0.071,True
4,A (1982-2021),Abuja,12,Hamon,NaN,-1.469,-1.409,-1.409,-0.059,-0.020,True
...,...,...,...,...,...,...,...,...,...,...,...
139,B (1985-2020),Yenagoa,12,Hamon,NaN,-0.751,-0.720,-0.720,-0.031,0.000,True
140,B (1985-2020),Yenagoa,12,Hargreaves,NaN,-0.702,-0.720,-0.720,0.017,0.000,False
141,B (1985-2020),Yenagoa,12,Oudin,NaN,-0.743,-0.720,-0.720,-0.023,0.000,True
142,B (1985-2020),Yenagoa,12,Penman-Monteith,NaN,-0.648,-0.720,-0.720,0.072,0.000,True


## SPEI - SPI versus temperature (Reviewer 3), Thornthwaite and Penman-Monteith

In [14]:
rows = []
for sc in (3, 12):
    for loc in P.columns:
        p = P[loc].loc['1984':]
        for pn in [n for n in ('Thornthwaite', 'Penman-Monteith') if n in PETS]:
            pet = PETS[pn][loc].loc['1984':]
            base = spei_cm(p, pd.Series(0.0, index=p.index), sc); spe = spei_cm(p, pet, sc)       # same rank-based method for both
            diff = (spe - base).dropna()
            tanom = (T[loc] - T[loc].groupby(T.index.month).transform('mean')).rolling(sc, min_periods=sc).mean()
            j = pd.concat([diff, tanom], axis=1, keys=['d', 't']).dropna().loc['1985':'2020']
            d_ = diff.loc['1985':'2020']; t_ = T[loc].loc['1985':'2020']
            md_, mt = mk_test(d_.values, hr=True), mk_test(t_.values, hr=True)
            rows.append(dict(PET=pn, Location=loc.title(), scale=sc, r_diff_vs_Tanomaly=round(np.corrcoef(j.d, j.t)[0, 1], 2),
                             diff_trend_per_decade=round(md_['sen'] * 120, 3), diff_trend_sig=md_['p'] < 0.05,
                             temp_trend_C_per_decade=round(mt['sen'] * 120, 3), temp_trend_sig=mt['p'] < 0.05))
TLINK = pd.DataFrame(rows)
print('SPEI minus a precipitation-only index built with the SAME method (so the gap is due to PET only). Correlations are descriptive.')
TLINK

SPEI minus a precipitation-only index built with the SAME method (so the gap is due to PET only). Correlations are descriptive.


,PET,Location,scale,r_diff_vs_Tanomaly,diff_trend_per_decade,diff_trend_sig,temp_trend_C_per_decade,temp_trend_sig
0,Thornthwaite,Abuja,3,-0.61,-0.076,True,0.544,True
1,Penman-Monteith,Abuja,3,-0.51,0.000,False,0.544,True
2,Thornthwaite,Gombe,3,-0.61,-0.071,True,0.100,False
3,Penman-Monteith,Gombe,3,-0.31,0.000,False,0.100,False
4,Thornthwaite,Ibadan,3,-0.80,0.000,False,0.068,False
5,Penman-Monteith,Ibadan,3,-0.11,0.011,True,0.068,False
6,Thornthwaite,Ikot Akpaden,3,-0.66,0.000,True,0.100,False
7,Penman-Monteith,Ikot Akpaden,3,-0.21,0.000,True,0.100,False
8,Thornthwaite,Jalingo,3,-0.62,-0.124,True,0.331,True
9,Penman-Monteith,Jalingo,3,-0.27,0.000,False,0.331,True


## Export

In [15]:
keep = ['window', 'PET', 'variant', 'location', 'scale', 'n_low', 'n_med', 'n_high', 'IISC', 'IISC_lo', 'IISC_hi', 'IISC_dir',
        'label_holm_low', 'label_holm_med', 'label_holm_high', 'A', 'A_lo', 'A_hi', 'A_clear']
with pd.ExcelWriter(os.path.join(OUT, 'SH-ITA_PET_sensitivity.xlsx'), engine='openpyxl') as w:
    PETCMP.to_excel(w, sheet_name='PET comparison', index=False)
    SENS[keep].round(3).to_excel(w, sheet_name='SH-ITA by PET and variant', index=False)
    AGR.to_excel(w, sheet_name='Agreement across PET', index=False)
    piv.round(3).to_excel(w, sheet_name='PET effect on IISC', index=False)
    TLINK.to_excel(w, sheet_name='SPEI vs P-only vs temp', index=False)
print('saved to', OUT)

saved to sh_ita_outputs
